# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [9]:
!pip install beir --no-deps


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [3]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip: 100%|██████████| 2.69M/2.69M [00:16<00:00, 175kiB/s] 


'../data/beir_datasets\\scifact'

In [4]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 42482.20it/s]


In [5]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [6]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [7]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [8]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [ ]:
from rank_bm25 import BM25Okapi
import nltk

# Texto de cada documento
documents = df_corpus["text"].fillna("").tolist()

# Tokenización simple
tokenized_corpus = [doc.lower().split() for doc in documents]

# Indice BM25
bm25 = BM25Okapi(tokenized_corpus)

# Mapeo posición -> doc_id
doc_ids = df_corpus["doc_id"].astype(str).tolist()

In [12]:
from tqdm import tqdm

bm25_results = {}

for qid, query in tqdm(queries.items()):
    
    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    # ordenar por score descendente
    ranked_idx = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )

    top_k = 100

    bm25_results[qid] = {
        doc_ids[i]: float(scores[i])
        for i in ranked_idx[:top_k]
    }

100%|██████████| 300/300 [00:08<00:00, 36.09it/s]


In [14]:
from beir.retrieval.evaluation import EvaluateRetrieval

retriever = EvaluateRetrieval()

ndcg, _map, recall, precision = retriever.evaluate(
    qrels,
    bm25_results,
    [10]
)

print("nDCG@10 =", ndcg["NDCG@10"])
print("Recall@10 =", recall["Recall@10"])

nDCG@10 = 0.54384
Recall@10 = 0.66883


In [15]:
qid = "133"

ranking = sorted(
    bm25_results[qid].items(),
    key=lambda x: x[1],
    reverse=True
)

print("Top 10 BM25")

for rank, (doc_id, score) in enumerate(ranking[:10], start=1):
    print(rank, doc_id, score)

Top 10 BM25
1 26688294 55.1964401863664
2 37964706 50.04011691148892
3 9507605 50.03740998262752
4 5270265 45.70320340871325
5 12785130 45.06239524984214
6 45764440 45.044936360734006
7 86694016 44.899837173478296
8 12640810 44.68953632297576
9 5821617 44.451933317595774
10 17934082 44.43642301215712


In [16]:
relevantes = set(
    df_qrels[df_qrels["query_id"] == qid]["doc_id"].astype(str)
)

for rank, (doc_id, score) in enumerate(ranking[:10], start=1):
    marca = "✓" if doc_id in relevantes else " "
    print(rank, marca, doc_id)

1   26688294
2   37964706
3   9507605
4   5270265
5   12785130
6   45764440
7   86694016
8 ✓ 12640810
9   5821617
10 ✓ 17934082


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [17]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

c:\Users\LabP5E004\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LabP5E004\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3617.35it/s]


In [21]:
TOP_K = 10
subset_qids = list(queries.keys())[:20]
print(subset_qids)

['1', '3', '5', '13', '36', '42', '48', '49', '50', '51', '53', '54', '56', '57', '70', '72', '75', '94', '99', '100']


In [39]:
reranked_results = {}

TOP_K = 100  # para pruebas

for qid in subset_qids:

    query = queries[qid]

    # candidatos recuperados por BM25
    candidates = list(bm25_results[qid].keys())[:TOP_K]

    pairs = [
        (query, corpus[doc_id]["text"])
        for doc_id in candidates
    ]

    ce_scores = cross_encoder.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    ranked_docs = sorted(
        zip(candidates, ce_scores),
        key=lambda x: x[1],
        reverse=True
    )

    reranked_results[qid] = {
        doc_id: float(score)
        for doc_id, score in ranked_docs
    }

### Comparación de antes y después.

In [29]:
qid = "13"


In [43]:

bm25_top10 = [
    doc_id
    for doc_id, score in sorted(
        bm25_results[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]
]

print("BM25")

for pos, doc_id in enumerate(bm25_top10, start=1):
    print(pos, doc_id)

BM25
1 26688294
2 37964706
3 9507605
4 5270265
5 12785130
6 45764440
7 86694016
8 12640810
9 5821617
10 17934082


In [44]:
for qid in reranked_results.keys():
    print(qid)

133


In [45]:


ce_top10 = [
    doc_id
    for doc_id, score in sorted(
        reranked_results[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]
]

print("\nCrossEncoder")

for pos, doc_id in enumerate(ce_top10, start=1):
    print(pos, doc_id)


CrossEncoder
1 16280642
2 12640810
3 35660758
4 36345185
5 6969753
6 21295300
7 9507605
8 17934082
9 86694016
10 19752008


### Identificar cambios de posición

In [46]:
print("Cambios de posición\n")

for doc_id in ce_top10:

    if doc_id in bm25_top10:

        pos_bm25 = bm25_top10.index(doc_id) + 1
        pos_ce = ce_top10.index(doc_id) + 1

        if pos_bm25 != pos_ce:

            print(
                f"Documento {doc_id}: "
                f"{pos_bm25} → {pos_ce}"
            )

Cambios de posición

Documento 12640810: 8 → 2
Documento 9507605: 3 → 7
Documento 17934082: 10 → 8
Documento 86694016: 7 → 9


Mostrar cuáles son relevantes

In [47]:
relevantes = set(
    df_qrels[
        df_qrels["query_id"] == qid
    ]["doc_id"].astype(str)
)

BM25

In [48]:
print("BM25")

for pos, doc_id in enumerate(bm25_top10, start=1):

    marca = "✓" if doc_id in relevantes else " "

    print(pos, marca, doc_id)

BM25
1   26688294
2   37964706
3   9507605
4   5270265
5   12785130
6   45764440
7   86694016
8 ✓ 12640810
9   5821617
10 ✓ 17934082


### Evaluar el nuevo ranking

In [49]:
from beir.retrieval.evaluation import EvaluateRetrieval

retriever = EvaluateRetrieval()

ndcg, _map, recall, precision = retriever.evaluate(
    qrels,
    reranked_results,
    [10]
)

In [50]:
print("CrossEncoder")

print("nDCG@10 =", ndcg["NDCG@10"])
print("MAP@10 =", _map["MAP@10"])
print("Recall@10 =", recall["Recall@10"])

CrossEncoder
nDCG@10 = 0.79134
MAP@10 = 0.62
Recall@10 = 0.8


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [51]:
import pandas as pd

rows = []

TOP_K = 100

for qid, retrieved_docs in bm25_results.items():

    query = queries[qid]

    relevant_docs = qrels.get(qid, {})

    for doc_id, bm25_score in list(retrieved_docs.items())[:TOP_K]:

        doc_text = corpus[doc_id]["text"]

        relevance = relevant_docs.get(doc_id, 0)

        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "bm25_score": bm25_score,
            "query_len": len(query.split()),
            "doc_len": len(doc_text.split()),
            "label": relevance
        })

ltr_df = pd.DataFrame(rows)

ltr_df.head()

,query_id,doc_id,bm25_score,query_len,doc_len,label
0,1,825728,9.634223,5,122,0
1,1,10931595,8.793938,5,216,0
2,1,13231899,7.416778,5,246,0
3,1,45638119,7.126155,5,147,0
4,1,17388232,7.118014,5,57,0


Crear matriz X e y

In [52]:
features = [
    "bm25_score",
    "query_len",
    "doc_len"
]

X = ltr_df[features]
y = ltr_df["label"]

In [54]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------------------------------ --- 1.3/1.5 MB 24.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 17.6 MB/s  0:00:00



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [68]:
from lightgbm import LGBMRanker

group = (
    ltr_df.groupby("query_id")
          .size()
          .tolist()
)

ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=100
)

ranker.fit(
    X,
    y,
    group=group
);

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000591 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 537
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 3


Re-rankear

In [56]:
ltr_df["ltr_score"] = ranker.predict(X)

ltr_results = {}

for qid, group_df in ltr_df.groupby("query_id"):

    ranked = group_df.sort_values(
        "ltr_score",
        ascending=False
    )

    ltr_results[qid] = {
        str(row["doc_id"]): float(row["ltr_score"])
        for _, row in ranked.iterrows()
    }

In [57]:
qid = "133"

Top 10 LTR

In [58]:
bm25_top10 = [
    doc
    for doc, score in sorted(
        bm25_results[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]
]

Top 10 LTR

In [59]:
ltr_top10 = [
    doc
    for doc, score in sorted(
        ltr_results[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]
]

Comparacion posiciones

In [60]:
for doc_id in ltr_top10:

    if doc_id in bm25_top10:

        old_pos = bm25_top10.index(doc_id) + 1
        new_pos = ltr_top10.index(doc_id) + 1

        if old_pos != new_pos:

            print(
                f"{doc_id}: "
                f"{old_pos} → {new_pos}"
            )

12640810: 8 → 2
17934082: 10 → 3
5270265: 4 → 6
37964706: 2 → 9


Evaluar

In [61]:
from beir.retrieval.evaluation import EvaluateRetrieval

retriever = EvaluateRetrieval()

ndcg, _map, recall, precision = retriever.evaluate(
    qrels,
    ltr_results,
    [10]
)

In [62]:
print("LTR")

print("nDCG@10 =", ndcg["NDCG@10"])
print("MAP@10 =", _map["MAP@10"])
print("Recall@10 =", recall["Recall@10"])

LTR
nDCG@10 = 0.77236
MAP@10 = 0.76567
Recall@10 = 0.77994


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

Evaluar BM25

In [63]:
from beir.retrieval.evaluation import EvaluateRetrieval

retriever = EvaluateRetrieval()

ndcg_bm25, map_bm25, recall_bm25, precision_bm25 = retriever.evaluate(
    qrels,
    bm25_results,
    [10]
)

In [64]:
print("BM25")
print("nDCG@10 =", ndcg_bm25["NDCG@10"])
print("MAP@10 =", map_bm25["MAP@10"])
print("Recall@10 =", recall_bm25["Recall@10"])

BM25
nDCG@10 = 0.54384
MAP@10 = 0.49926
Recall@10 = 0.66883


Evaluar Cross-Encoder

In [65]:
ndcg_ce, map_ce, recall_ce, precision_ce = retriever.evaluate(
    qrels,
    reranked_results,
    [10]
)

print("CrossEncoder")
print("nDCG@10 =", ndcg_ce["NDCG@10"])
print("MAP@10 =", map_ce["MAP@10"])
print("Recall@10 =", recall_ce["Recall@10"])

CrossEncoder
nDCG@10 = 0.79134
MAP@10 = 0.62
Recall@10 = 0.8


Evaluar LTR

In [66]:
ndcg_ltr, map_ltr, recall_ltr, precision_ltr = retriever.evaluate(
    qrels,
    ltr_results,
    [10]
)

print("LTR")
print("nDCG@10 =", ndcg_ltr["NDCG@10"])
print("MAP@10 =", map_ltr["MAP@10"])
print("Recall@10 =", recall_ltr["Recall@10"])

LTR
nDCG@10 = 0.77236
MAP@10 = 0.76567
Recall@10 = 0.77994


Tabla comparativa

In [67]:
import pandas as pd

results = pd.DataFrame({
    "Metodo": ["BM25", "CrossEncoder", "LTR"],
    "nDCG@10": [
        ndcg_bm25["NDCG@10"],
        ndcg_ce["NDCG@10"],
        ndcg_ltr["NDCG@10"]
    ],
    "MAP@10": [
        map_bm25["MAP@10"],
        map_ce["MAP@10"],
        map_ltr["MAP@10"]
    ],
    "Recall@10": [
        recall_bm25["Recall@10"],
        recall_ce["Recall@10"],
        recall_ltr["Recall@10"]
    ]
})

results

,Metodo,nDCG@10,MAP@10,Recall@10
0,BM25,0.54384,0.49926,0.66883
1,CrossEncoder,0.79134,0.62000,0.80000
2,LTR,0.77236,0.76567,0.77994
